In [1]:
from rich import columns
from torch.cuda import is_available as is_cuda_available
from lighthouse.models import CGDETRPredictor
from utils.helper_methods import split_video_ffmpeg, parse_timestamps, extract_clips
import pandas
import os

C:\Users\denve\miniconda3\envs\moment-retrieval-and-video-summarization\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model weights can be downloaded from https://zenodo.org/records/13960580
Checkpoint file should correspond to the feature selected in predictor

For slowfast video encoding features additional weights should be downloaded. They can be found in lighthouse GitHub page (https://github.com/line/lighthouse/tree/main) or downloaded from [here](https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl)

In [2]:
device = "cuda" if is_cuda_available() else "cpu"
device

'cuda'

In [3]:
# model = CGDETRPredictor('model_weights/clip_cg_detr_qvhighlight.ckpt', device=device,
#                         feature_name='clip')

model = CGDETRPredictor('model_weights/clip_slowfast_cg_detr_qvhighlight.ckpt', device=device,
                        feature_name='clip_slowfast', slowfast_path="model_weights/SLOWFAST_8x8_R50.pkl", pann_path=None)

Use path to a folder with video segments, not exceeding 150 seconds(due to limitation of the model). ```split_video_ffmpeg()``` can be used to split the video into chunks

In [4]:
chunks_path = "TVSum/video_25_chunks"
# chunks_path = split_video_ffmpeg(video_path, 150)

Generating results using human annotations for the video

In [5]:
annotations = pandas.read_csv("Annotations/7-TVSum-video_25-i6324060.txt")
annotations['timestamp'] = annotations['timestamp'].map(parse_timestamps)

results = []

for row in annotations.iterrows():
    query, timestamp, subjectivity_rating = row[1]
    query_results = []
    for index, chunk in enumerate(os.listdir(chunks_path)):
        if chunk.endswith('.mp4'):
            video_path = chunks_path + '/' + chunk
            video = model.encode_video(video_path)
            query_results.append((index, model.predict(query, video)))

    results.append({
        "query": query,
        "true_timestamp": timestamp,
        "predictions": query_results
    })

In [6]:
for query_result in results:
    print(f"Query: {query_result['query']}")
    print(f"True timestamp: {query_result['true_timestamp']}")


    for chunk in query_result['predictions']:
        print("\tChunk:", chunk[0])
        print(f"\t\tClosest Moment: {chunk[1]['pred_relevant_windows'][0][0], chunk[1]['pred_relevant_windows'][0][1]}, Corresponding Saliency score: {chunk[1]['pred_relevant_windows'][0][2]}")
    print("-" * 50)

Query: Two young men are traveling on a train and recording each other with cameras
True timestamp: (3.0, 6.0)
	Chunk: 0
		Closest Moment: (2.6967, 8.1025), Corresponding Saliency score: 0.9879
	Chunk: 1
		Closest Moment: (87.9291, 100.7757), Corresponding Saliency score: 0.9994
--------------------------------------------------
Query: Short clips with landscape views from a train window are shown
True timestamp: (7.0, 12.0)
	Chunk: 0
		Closest Moment: (132.779, 144.7218), Corresponding Saliency score: 0.9916
	Chunk: 1
		Closest Moment: (0.0, 8.5486), Corresponding Saliency score: 0.9993
--------------------------------------------------
Query: A group of young people are warming up together
True timestamp: (27.0, 37.0)
	Chunk: 0
		Closest Moment: (24.9719, 54.2116), Corresponding Saliency score: 0.9969
	Chunk: 1
		Closest Moment: (23.978, 50.8973), Corresponding Saliency score: 0.9931
--------------------------------------------------
Query: A young man performs parkour maneuvers over

Using custom query prompts

In [7]:
events = ["A person walking down a street in a city, passing by various buildings and shops",
          "A person sitting on the ground, wearing earphones and looking at something in their hands",
          "A group of people gathered around a banner, with one person holding a microphone",
          "The video then shows a group of people standing on a beach, with a large building in the background",
          "A person lying on a bed, holding a camera",
          "A group of people playing in a dirt field, with a large building in the background"]

results = []

for query in events:
    query_results = []
    for index, chunk in enumerate(os.listdir(chunks_path)):
        if chunk.endswith('.mp4'):
            video_path = chunks_path + '/' + chunk
            video = model.encode_video(video_path)
            query_results.append((index, model.predict(query, video)))

    results.append({
        "query": query,
        "predictions": query_results
    })

In [8]:
for query_result in results:
    print(f"Query: {query_result['query']}")
    # print(f"True timestamp: {query_result['true_timestamp']}")


    for chunk in query_result['predictions']:
        print("\tChunk:", chunk[0])
        print(f"\t\tClosest Moment: {chunk[1]['pred_relevant_windows'][0][0], chunk[1]['pred_relevant_windows'][0][1]}, Corresponding Saliency score: {chunk[1]['pred_relevant_windows'][0][2]}")
    print("-" * 50)

Query: A person walking down a street in a city, passing by various buildings and shops
	Chunk: 0
		Closest Moment: (75.1742, 90.677), Corresponding Saliency score: 0.9996
	Chunk: 1
		Closest Moment: (56.7895, 69.4674), Corresponding Saliency score: 0.9993
--------------------------------------------------
Query: A person sitting on the ground, wearing earphones and looking at something in their hands
	Chunk: 0
		Closest Moment: (91.4469, 97.8372), Corresponding Saliency score: 0.9812
	Chunk: 1
		Closest Moment: (79.012, 98.4856), Corresponding Saliency score: 0.9998
--------------------------------------------------
Query: A group of people gathered around a banner, with one person holding a microphone
	Chunk: 0
		Closest Moment: (22.084, 32.724), Corresponding Saliency score: 0.9712
	Chunk: 1
		Closest Moment: (21.7495, 32.5707), Corresponding Saliency score: 0.9432
--------------------------------------------------
Query: The video then shows a group of people standing on a beach, w

Manually clipping the video according to the returned timestamps

In [9]:
timecodes = [(22.084, 32.724),
             (75.1742, 90.677),
             (91.4469, 97.8372),
             (172.6741, 184.0032),
             (189.3924, 202.1697),
             (228.6148, 241.6699)
             ]
extract_clips("TVSum/video_25.mp4", "video_25_summary.mp4", timecodes)

Processing clips... this may take a moment.
Success! Video saved as video_25_summary.mp4
